# 📊 ACIS Insurance — A/B Hypothesis Testing
## Task 3: Statistical Validation of Risk Drivers
---
**Decision Rule:** Reject H₀ when **p-value < 0.05**

| # | Null Hypothesis | Test | KPI |
|---|---|---|---|
| H1 | No risk differences across provinces | Z-test | Claim Frequency |
| H2 | No risk differences between zip codes | Z-test | Claim Severity |
| H3 | No margin difference between zip codes | Z-test | Margin |
| H4 | No risk difference between genders | Chi-squared | Claim Frequency |

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from src.data_loader import load_and_prepare
from src.hypothesis_tests import (
    test_claim_frequency, test_claim_severity,
    test_margin, test_chi_squared, print_results
)
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Imports successful')

In [ ]:
DATA_PATH = '../data/MachineLearningRating_v3.txt'
df = load_and_prepare(DATA_PATH)
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
print(f'Dataset: {df.shape[0]:,} rows')
print(f'Policies with claims: {df["HasClaim"].sum():,} ({df["HasClaim"].mean():.2%})')

---
## H1: Province Risk
**Control:** Northern Cape | **Test:** Gauteng

In [ ]:
gauteng = df[df['Province'] == 'Gauteng']['TotalClaims']
northern_cape = df[df['Province'] == 'Northern Cape']['TotalClaims']
h1_results = test_claim_frequency(northern_cape, gauteng, 'Northern Cape', 'Gauteng')
print_results(h1_results)

In [ ]:
province_freq = df.groupby('Province').agg(
    PolicyCount=('TotalClaims', 'count'),
    ClaimCount=('HasClaim', 'sum')
).reset_index()
province_freq['ClaimFrequency'] = province_freq['ClaimCount'] / province_freq['PolicyCount']
province_freq = province_freq.sort_values('ClaimFrequency', ascending=False)
overall_freq = df['HasClaim'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if f > overall_freq else '#2ecc71'
          for f in province_freq['ClaimFrequency']]
bars = ax.bar(province_freq['Province'], province_freq['ClaimFrequency'],
              color=colors, edgecolor='white')
ax.axhline(y=overall_freq, color='navy', linestyle='--', linewidth=2,
           label=f'Portfolio Avg ({overall_freq:.4%})')
ax.set_title('Claim Frequency by Province\n(Red = above average, Green = below average)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Claim Frequency')
ax.legend()
plt.xticks(rotation=45, ha='right')
for bar, val in zip(bars, province_freq['ClaimFrequency']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.00002,
            f'{val:.4%}', ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/h1_province_claim_freq.png', dpi=150, bbox_inches='tight')
plt.show()

### 📝 H1 Result: REJECT H₀ (p = 0.003793)
Gauteng (0.3356%) is 167.68% more likely to result in a claim than Northern Cape (0.1254%).
**Recommendation:** Province-level premium adjustments are statistically justified.

---
## H2: Zip Code Claim Severity
**Groups:** Top 25% vs Bottom 25% severity zip codes (min 100 policies, 1 claim)

In [ ]:
zip_stats = df.groupby('PostalCode').agg(
    PolicyCount=('TotalClaims', 'count'),
    ClaimCount=('HasClaim', 'sum'),
    AvgSeverity=('TotalClaims', lambda x: x[x > 0].mean())
).reset_index()
zip_stats = zip_stats[
    (zip_stats['PolicyCount'] >= 100) & (zip_stats['ClaimCount'] >= 1)
]
zip_stats['ClaimFrequency'] = zip_stats['ClaimCount'] / zip_stats['PolicyCount']
print(f'Qualifying zip codes: {len(zip_stats)}')

high_sev_threshold = zip_stats['AvgSeverity'].quantile(0.75)
low_sev_threshold = zip_stats['AvgSeverity'].quantile(0.25)
high_sev_zips = zip_stats[zip_stats['AvgSeverity'] >= high_sev_threshold]['PostalCode']
low_sev_zips = zip_stats[zip_stats['AvgSeverity'] <= low_sev_threshold]['PostalCode']
print(f'High-severity zips: {len(high_sev_zips)} | Low-severity zips: {len(low_sev_zips)}')

high_sev_claims = df[df['PostalCode'].isin(high_sev_zips)]['TotalClaims']
low_sev_claims = df[df['PostalCode'].isin(low_sev_zips)]['TotalClaims']
h2_results = test_claim_severity(
    low_sev_claims, high_sev_claims,
    'Low-Severity Zips (bottom 25%)', 'High-Severity Zips (top 25%)'
)
print_results(h2_results)

### 📝 H2 Result: REJECT H₀ (p < 0.000001)
High-severity zip codes: avg claim R45,525 vs R1,724 in low-severity zips — a 2,540% difference.
**Recommendation:** 4-tier zip code severity pricing system required.

---
## H3: Zip Code Margin
**Groups:** Same high-severity vs low-severity zip codes as H2

In [ ]:
high_sev_margin = df[df['PostalCode'].isin(high_sev_zips)]['Margin']
low_sev_margin = df[df['PostalCode'].isin(low_sev_zips)]['Margin']
h3_results = test_margin(
    low_sev_margin, high_sev_margin,
    'Low-Severity Zip Codes', 'High-Severity Zip Codes'
)
print_results(h3_results)

### 📝 H3 Result: REJECT H₀ (p < 0.000001)
Low-risk zips earn R+59.93/policy. High-risk zips lose R-140.20/policy. Gap: R200.13 per policy.
**Recommendation:** PostalCode is the strongest profitability driver — must be in pricing model.

---
## H4: Gender Risk
**Note:** 94.1% of gender data is missing — results must be interpreted with caution.

In [ ]:
gender_df = df[df['Gender'].isin(['Male', 'Female'])]
print(f'Records with known gender: {len(gender_df):,}')
print(f'Male: {(gender_df["Gender"]=="Male").sum():,} | Female: {(gender_df["Gender"]=="Female").sum():,}')
male_claims = gender_df[gender_df['Gender'] == 'Male']['TotalClaims']
female_claims = gender_df[gender_df['Gender'] == 'Female']['TotalClaims']
h4_results = test_chi_squared(male_claims, female_claims, 'Male', 'Female')
print_results(h4_results)

In [ ]:
gender_stats = gender_df.groupby('Gender').agg(
    PolicyCount=('TotalClaims', 'count'),
    ClaimCount=('HasClaim', 'sum'),
    AvgClaim=('TotalClaims', lambda x: x[x > 0].mean())
).reset_index()
gender_stats['ClaimFrequency'] = gender_stats['ClaimCount'] / gender_stats['PolicyCount']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Risk Comparison by Gender', fontsize=14, fontweight='bold')
axes[0].bar(gender_stats['Gender'], gender_stats['ClaimFrequency'],
            color=['#3498db', '#e74c3c'], edgecolor='white', alpha=0.85)
axes[0].set_title('Claim Frequency by Gender')
axes[0].set_ylabel('Claim Frequency')
for i, (_, row) in enumerate(gender_stats.iterrows()):
    axes[0].text(i, row['ClaimFrequency'] + 0.00005,
                 f"{row['ClaimFrequency']:.4%}", ha='center', fontweight='bold')
axes[1].bar(gender_stats['Gender'], gender_stats['AvgClaim'],
            color=['#3498db', '#e74c3c'], edgecolor='white', alpha=0.85)
axes[1].set_title('Average Claim Severity by Gender')
axes[1].set_ylabel('Avg Claim Amount (Rand)')
plt.tight_layout()
plt.savefig('../reports/h4_gender_risk.png', dpi=150, bbox_inches='tight')
plt.show()

### 📝 H4 Result: FAIL TO REJECT H₀ (p = 0.9515)
Male (0.2195%) vs Female (0.2073%) — negligible difference, 95% probability it is random noise.
**Recommendation:** Do NOT use gender as a pricing variable. Statistically insignificant + 94% missing data + regulatory risk.

---
## 📋 Final Results Summary

| # | Hypothesis | KPI | Test | p-value | Decision |
|---|---|---|---|---|---|
| H1 | Province risk | Claim Frequency | Z-test | 0.003793 | ✅ REJECT H₀ |
| H2 | Zip code severity | Claim Severity | Welch t-test | < 0.000001 | ✅ REJECT H₀ |
| H3 | Zip code margin | Margin (R) | Welch t-test | < 0.000001 | ✅ REJECT H₀ |
| H4 | Gender risk | Claim Frequency | Chi-squared | 0.951464 | ❌ FAIL TO REJECT H₀ |

**Hypotheses Rejected: 3/4**

| Feature | Evidence | Task 4 Priority |
|---|---|---|
| PostalCode | H2+H3 rejected (p<0.000001) | ✅ Highest |
| Province | H1 rejected (p=0.0038) | ✅ High |
| Gender | H4 NOT rejected (p=0.9515) | ❌ Exclude |

In [ ]:
results_summary = pd.DataFrame([
    {'Hypothesis': 'H1', 'p-value': 0.003793,
     'Decision': 'REJECT H0', 'Key Finding': 'Gauteng 167% higher freq than N.Cape'},
    {'Hypothesis': 'H2', 'p-value': 0.000000,
     'Decision': 'REJECT H0', 'Key Finding': 'R43,801 severity gap per claim'},
    {'Hypothesis': 'H3', 'p-value': 0.000000,
     'Decision': 'REJECT H0', 'Key Finding': 'R200.13 margin gap per policy'},
    {'Hypothesis': 'H4', 'p-value': 0.951464,
     'Decision': 'FAIL TO REJECT H0', 'Key Finding': 'No meaningful gender difference'},
])
print('HYPOTHESIS TESTING COMPLETE')
print('=' * 65)
print(results_summary.to_string(index=False))
print('=' * 65)
print(f'Hypotheses rejected: {(results_summary["Decision"]=="REJECT H0").sum()}/4')